# Publication-Ready Fraud Detection Benchmark

Colab-ready notebook for the current paper. It uses a leak-proof flow:

raw data -> train/validation/test split -> fit preprocessing on train only -> SMOTE/ADASYN on train only -> train baselines -> tune threshold on validation -> final test evaluation -> cost, top-k, latency, bootstrap CI, calibration, temporal drift, McNemar-Holm, and SHAP.

Outputs are saved under Google Drive:
`/content/drive/MyDrive/Downloads/Credit/Fraud_Detection_Q1_Publication_Ready/`

## 1. Install Dependencies

In [ ]:
!pip -q install xgboost lightgbm catboost imbalanced-learn shap statsmodels psutil openpyxl

## 2. Mount Drive and Configure Output Folders

In [ ]:
from google.colab import drive
from pathlib import Path
import os, json, time, gc, math, warnings, platform, subprocess
warnings.filterwarnings("ignore")

drive.mount("/content/drive")

PROJECT_DIR = Path("/content/drive/MyDrive/Downloads/Credit/Fraud_Detection_Q1_Publication_Ready")
DATA_SOURCE_ROOT = Path("/content/drive/MyDrive/Credit-Card-new1/Datasets")
DATA_DIR_CANDIDATES = [
    DATA_SOURCE_ROOT / "Datasets" / "Datasets",
    DATA_SOURCE_ROOT / "Datasets",
    DATA_SOURCE_ROOT,
    Path("/content/drive/MyDrive/CCC"),
]
REQUIRED_FILES = ["creditcard.csv", "fraudTest.csv", "PS.csv"]
DATA_DIR = next((p for p in DATA_DIR_CANDIDATES if all((p / f).exists() for f in REQUIRED_FILES)), None)
if DATA_DIR is None:
    raise FileNotFoundError("Could not find the three CSV files. Checked: " + str([str(p) for p in DATA_DIR_CANDIDATES]))

RUN_ID = time.strftime("%Y%m%d_%H%M%S")
OUT_DIR = PROJECT_DIR
TABLE_DIR = OUT_DIR / "01_tables_csv"
FIG_DIR = OUT_DIR / "02_figures_png_pdf"
MODEL_DIR = OUT_DIR / "03_models"
LOG_DIR = OUT_DIR / "04_logs"
for d in [OUT_DIR, FIG_DIR, TABLE_DIR, MODEL_DIR, LOG_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print("Dataset folder:", DATA_DIR)
print("Output folder:", OUT_DIR)

## 3. Imports, Settings, and Figure Style

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import psutil, joblib
from scipy import sparse
from scipy.stats import ks_2samp

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import RobustScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, average_precision_score, matthews_corrcoef, confusion_matrix,
    roc_curve, precision_recall_curve, brier_score_loss)
from sklearn.calibration import calibration_curve
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

from imblearn.over_sampling import SMOTE, ADASYN
from imblearn.combine import SMOTETomek
from imblearn.ensemble import BalancedRandomForestClassifier, EasyEnsembleClassifier

import xgboost as xgb
import lightgbm as lgb
from catboost import CatBoostClassifier
from statsmodels.stats.contingency_tables import mcnemar
from statsmodels.stats.multitest import multipletests
import shap

try:
    import torch
    import torch.nn as nn
    import torch.optim as optim
    from torch.utils.data import DataLoader, TensorDataset
    TORCH_OK = True
except Exception as e:
    TORCH_OK = False
    print("Torch unavailable; MLP skipped:", e)

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

# Colab Pro T4 settings. For final paper run keep these as-is.
RUN_DEEP_MLP = True
RUN_SMOTE_TOMEK = False       # Set True only for an extended final run; it can be slow.
BOOTSTRAP_ROUNDS = 300        # Use 500-1000 if you have extra time.
SHAP_SAMPLE_SIZE = 500
LATENCY_WARMUP_RUNS = 20
LATENCY_REPEATS = 100
LATENCY_BATCH_SIZE = 4096
TEST_SIZE = 0.20
VALIDATION_SIZE = 0.20
MAX_ROWS = {"D1_Kaggle_CC": None, "D2_Online_Fraud": 300_000, "D3_PaySim": 200_000}
FN_COSTS = [10, 50, 100, 500]
PRIMARY_FN_COST, FP_COST = 100, 1
TOP_K_PCTS = [0.01, 0.02, 0.05, 0.10]
MAIN_MODEL = "XGB_SMOTE"

MODEL_DISPLAY = {
    "LR_balanced": "LogReg balanced", "RF": "Random Forest", "BRF": "Balanced RF",
    "EasyEnsemble": "EasyEnsemble", "XGB": "XGBoost", "XGB_spw": "XGBoost scale_pos_weight",
    "XGB_SMOTE": "XGBoost + SMOTE", "XGB_ADASYN": "XGBoost + ADASYN",
    "XGB_SMOTETomek": "XGBoost + SMOTE-Tomek", "LightGBM": "LightGBM balanced",
    "CatBoost": "CatBoost balanced", "MLP_Focal": "MLP + Focal Loss",
    "MLP_Focal_SMOTE": "MLP + Focal + SMOTE"
}

plt.rcParams.update({
    "figure.dpi": 150, "savefig.dpi": 300, "font.size": 10, "axes.titlesize": 11,
    "axes.labelsize": 10, "xtick.labelsize": 8, "ytick.labelsize": 8,
    "legend.fontsize": 8, "figure.facecolor": "white", "axes.facecolor": "white",
    "axes.grid": True, "grid.alpha": 0.25, "axes.spines.top": False, "axes.spines.right": False
})
sns.set_theme(style="whitegrid", context="paper")

def savefig(name):
    png, pdf = FIG_DIR / f"{name}.png", FIG_DIR / f"{name}.pdf"
    plt.savefig(png, dpi=300, bbox_inches="tight")
    plt.savefig(pdf, bbox_inches="tight")
    print("Saved", png)

def shell_output(cmd):
    try:
        return subprocess.check_output(cmd, shell=True, text=True, stderr=subprocess.STDOUT).strip()
    except Exception as e:
        return f"Unavailable: {e}"

env_info = {
    "run_id": RUN_ID, "data_dir": str(DATA_DIR), "output_dir": str(OUT_DIR),
    "ram_gb": round(psutil.virtual_memory().total / (1024**3), 2),
    "gpu_name": shell_output("nvidia-smi --query-gpu=name --format=csv,noheader 2>/dev/null"),
    "python_version": platform.python_version(), "xgboost_version": xgb.__version__,
    "lightgbm_version": lgb.__version__, "run_deep_mlp": RUN_DEEP_MLP,
    "run_smote_tomek": RUN_SMOTE_TOMEK, "bootstrap_rounds": BOOTSTRAP_ROUNDS
}
pd.DataFrame([env_info]).to_csv(TABLE_DIR / "00_environment.csv", index=False)
with open(LOG_DIR / "00_environment.json", "w") as f:
    json.dump(env_info, f, indent=2)
pd.DataFrame([env_info]).T

## 4. Leak-Proof Data Preparation

In [ ]:
DATASET_SPECS = {
    "D1_Kaggle_CC": {"file": "creditcard.csv", "label": "Class", "time": ["Time"]},
    "D2_Online_Fraud": {"file": "fraudTest.csv", "label": "is_fraud", "time": ["unix_time", "trans_date_trans_time"]},
    "D3_PaySim": {"file": "PS.csv", "label": "isFraud", "time": ["step"]},
}
ID_COLS = {"Unnamed: 0", "cc_num", "first", "last", "street", "zip", "dob", "trans_num",
           "nameOrig", "nameDest", "customer_id", "account_id"}
LEAK_COLS = {"isFlaggedFraud"}
HIGH_CARD_KEEP = {"merchant"}

def read_csv(path):
    try:
        return pd.read_csv(path)
    except UnicodeDecodeError:
        return pd.read_csv(path, encoding="latin1")

def add_datetime_features(df):
    df = df.copy()
    for c in list(df.columns):
        if ("date" in c.lower() or "time" in c.lower()) and not pd.api.types.is_numeric_dtype(df[c]):
            s = pd.to_datetime(df[c], errors="coerce")
            if s.notna().mean() >= 0.80:
                df[c + "_hour"] = s.dt.hour.fillna(-1).astype(int)
                df[c + "_dayofweek"] = s.dt.dayofweek.fillna(-1).astype(int)
                df[c + "_month"] = s.dt.month.fillna(-1).astype(int)
                df = df.drop(columns=[c])
    return df

def choose_time_col(df, candidates):
    return next((c for c in candidates if c in df.columns), None)

def sample_keep_all_fraud(df, label, max_rows):
    if max_rows is None or len(df) <= max_rows:
        return df.copy()
    fraud, normal = df[df[label] == 1], df[df[label] == 0]
    n_normal = max(0, min(max_rows - len(fraud), len(normal)))
    return pd.concat([fraud, normal.sample(n=n_normal, random_state=RANDOM_STATE)]).sample(frac=1, random_state=RANDOM_STATE).reset_index(drop=True)

def drop_problem_columns(Xtr, Xv, Xte):
    drop = set()
    for c in Xtr.columns:
        if c in ID_COLS or c in LEAK_COLS:
            drop.add(c)
        elif Xtr[c].dtype == "object" and Xtr[c].nunique(dropna=True) > 300 and c not in HIGH_CARD_KEEP:
            drop.add(c)
    return Xtr.drop(columns=list(drop), errors="ignore"), Xv.drop(columns=list(drop), errors="ignore"), Xte.drop(columns=list(drop), errors="ignore"), sorted(drop)

def make_preprocessor(Xtr):
    num = Xtr.select_dtypes(include=["number", "bool"]).columns.tolist()
    cat = [c for c in Xtr.columns if c not in num]
    try:
        enc = OneHotEncoder(handle_unknown="ignore", min_frequency=50, sparse_output=True)
    except TypeError:
        enc = OneHotEncoder(handle_unknown="ignore", min_frequency=50, sparse=True)
    return ColumnTransformer([
        ("num", Pipeline([("imputer", SimpleImputer(strategy="median")), ("scaler", RobustScaler(with_centering=False))]), num),
        ("cat", Pipeline([("imputer", SimpleImputer(strategy="most_frequent")), ("onehot", enc)]), cat)
    ], sparse_threshold=0.30, remainder="drop")

def get_feature_names(prep, n):
    try:
        return np.array([str(x).replace("num__", "").replace("cat__", "") for x in prep.get_feature_names_out()])
    except Exception:
        return np.array([f"feature_{i}" for i in range(n)])

def as_f32(X):
    return X.astype(np.float32) if sparse.issparse(X) else np.asarray(X, dtype=np.float32)

prepared, dataset_rows, class_before, class_after = {}, [], [], []

for name, spec in DATASET_SPECS.items():
    df = read_csv(DATA_DIR / spec["file"]).drop_duplicates().reset_index(drop=True)
    label = spec["label"]
    df[label] = df[label].astype(int)
    original_rows = len(df)
    df = sample_keep_all_fraud(df, label, MAX_ROWS[name])
    raw_for_time = df.copy()
    time_col = choose_time_col(raw_for_time, spec["time"])
    df = add_datetime_features(df)
    X, y = df.drop(columns=[label]), df[label].astype(int)

    X_train_raw, X_test_raw, y_train, y_test = train_test_split(X, y, test_size=TEST_SIZE, stratify=y, random_state=RANDOM_STATE)
    X_fit_raw, X_val_raw, y_fit, y_val = train_test_split(X_train_raw, y_train, test_size=VALIDATION_SIZE, stratify=y_train, random_state=RANDOM_STATE)
    X_fit_raw, X_val_raw, X_test_raw, dropped = drop_problem_columns(X_fit_raw, X_val_raw, X_test_raw)

    prep = make_preprocessor(X_fit_raw)
    X_fit, X_val, X_test = as_f32(prep.fit_transform(X_fit_raw)), as_f32(prep.transform(X_val_raw)), as_f32(prep.transform(X_test_raw))
    names = get_feature_names(prep, X_fit.shape[1])
    prepared[name] = dict(source_file=spec["file"], raw_for_time=raw_for_time, label=label, time_col=time_col,
                          dropped=dropped, X_fit=X_fit, X_val=X_val, X_test=X_test,
                          y_fit=y_fit.reset_index(drop=True), y_val=y_val.reset_index(drop=True),
                          y_test=y_test.reset_index(drop=True), preprocessor=prep, feature_names=names)
    dataset_rows.append(dict(dataset=name, source_file=spec["file"], original_rows=original_rows, rows_used=len(y),
        fraud_cases=int(y.sum()), fraud_rate_pct=float(y.mean()*100), fit_rows=len(y_fit),
        validation_rows=len(y_val), test_rows=len(y_test), test_fraud_cases=int(y_test.sum()),
        features_after_preprocessing=int(X_fit.shape[1]), time_column_for_drift=time_col,
        dropped_columns=", ".join(dropped)))
    for split, yy in [("fit_train", y_fit), ("validation", y_val), ("test", y_test)]:
        class_before += [dict(dataset=name, split=split, cls="non_fraud", count=int((yy == 0).sum())),
                         dict(dataset=name, split=split, cls="fraud", count=int((yy == 1).sum()))]
    print(f"{name}: rows={len(y):,}, fraud={int(y.sum()):,}, features={X_fit.shape[1]}, time_col={time_col}")

dataset_table = pd.DataFrame(dataset_rows)
dataset_table.to_csv(TABLE_DIR / "01_dataset_audit.csv", index=False)
pd.DataFrame(class_before).to_csv(TABLE_DIR / "02_class_distribution_before_resampling.csv", index=False)
dataset_table

## 5. Workflow and Class Balance Figures

In [ ]:
fig, ax = plt.subplots(figsize=(13, 3.2)); ax.axis("off")
steps = ["Raw\nDatasets", "Split\nFirst", "Preprocess\nTrain Only", "SMOTE/ADASYN\nTrain Only",
         "Train\nBaselines", "Tune Threshold\nValidation", "Final\nTest", "SHAP/Drift/\nStats"]
x = np.linspace(0.05, 0.95, len(steps))
for i, (xi, lab) in enumerate(zip(x, steps)):
    ax.text(xi, .55, lab, ha="center", va="center", fontsize=9,
            bbox=dict(boxstyle="round,pad=.35", fc="#F7F7F7", ec="#333333"))
    if i < len(steps)-1:
        ax.annotate("", xy=(x[i+1]-.055,.55), xytext=(xi+.055,.55), arrowprops=dict(arrowstyle="->", lw=1.4))
ax.set_title("Leak-proof and deployment-aware experimental workflow")
savefig("01_research_workflow"); plt.show()

before = pd.DataFrame(class_before)
plt.figure(figsize=(8.5, 4.8))
sns.barplot(data=before[before["split"] == "fit_train"], x="dataset", y="count", hue="cls", palette=["#4C78A8", "#F58518"])
plt.yscale("log"); plt.xlabel(""); plt.ylabel("Count (log scale)")
plt.title("Training class distribution before resampling"); plt.xticks(rotation=15, ha="right")
plt.tight_layout(); savefig("02_class_distribution_before_resampling"); plt.show()

## 6. Apply SMOTE/ADASYN Only to Training Data

In [ ]:
resampled, resampling_rows = {}, []
for name, ds in prepared.items():
    resampled[name] = {"original": (ds["X_fit"], ds["y_fit"])}
    samplers = {"SMOTE": SMOTE(random_state=RANDOM_STATE, k_neighbors=5),
                "ADASYN": ADASYN(random_state=RANDOM_STATE, n_neighbors=5)}
    if RUN_SMOTE_TOMEK:
        samplers["SMOTETomek"] = SMOTETomek(random_state=RANDOM_STATE, smote=SMOTE(random_state=RANDOM_STATE, k_neighbors=5))
    for sname, sampler in samplers.items():
        t0 = time.perf_counter()
        try:
            Xr, yr = sampler.fit_resample(ds["X_fit"], ds["y_fit"])
            Xr, yr = as_f32(Xr), pd.Series(yr).astype(int).reset_index(drop=True)
            resampled[name][sname] = (Xr, yr); status = "ok"
        except Exception as e:
            Xr, yr, status = None, None, f"failed: {e}"
        sec = time.perf_counter() - t0
        resampling_rows.append(dict(dataset=name, resampler=sname, status=status, seconds=sec,
            rows_before=len(ds["y_fit"]), fraud_before=int((ds["y_fit"]==1).sum()),
            rows_after=len(yr) if yr is not None else np.nan, fraud_after=int((yr==1).sum()) if yr is not None else np.nan))
        if yr is not None:
            class_after.extend([dict(dataset=name, resampler=sname, cls="non_fraud", count=int((yr==0).sum())),
                                dict(dataset=name, resampler=sname, cls="fraud", count=int((yr==1).sum()))])
        print(name, sname, status, f"{sec:.1f}s")
resampling_table = pd.DataFrame(resampling_rows)
resampling_table.to_csv(TABLE_DIR / "03_training_only_resampling_audit.csv", index=False)
pd.DataFrame(class_after).to_csv(TABLE_DIR / "04_class_distribution_after_resampling.csv", index=False)

after = pd.DataFrame(class_after)
if len(after):
    plt.figure(figsize=(9.5, 4.8))
    sns.barplot(data=after[after["resampler"].isin(["SMOTE", "ADASYN"])], x="dataset", y="count", hue="cls", palette=["#4C78A8", "#F58518"])
    plt.xlabel(""); plt.ylabel("Count"); plt.title("Training class distribution after resampling")
    plt.xticks(rotation=15, ha="right"); plt.tight_layout(); savefig("03_class_distribution_after_resampling"); plt.show()
resampling_table

## 7. Evaluation Helper Functions

In [ ]:
def proba(model, X):
    if hasattr(model, "predict_proba"):
        return np.asarray(model.predict_proba(X)[:, 1], dtype=float)
    scores = np.asarray(model.decision_function(X), dtype=float)
    return 1 / (1 + np.exp(-scores))

def metrics(y, p, thr=.5, fn_cost=PRIMARY_FN_COST, fp_cost=FP_COST):
    y = np.asarray(y).astype(int); p = np.asarray(p, dtype=float); pred = (p >= thr).astype(int)
    tn, fp, fn, tp = confusion_matrix(y, pred, labels=[0,1]).ravel(); n = len(y)
    return dict(threshold=float(thr), accuracy=accuracy_score(y,pred), precision=precision_score(y,pred,zero_division=0),
        recall=recall_score(y,pred,zero_division=0), f1=f1_score(y,pred,zero_division=0), mcc=matthews_corrcoef(y,pred),
        roc_auc=roc_auc_score(y,p) if len(np.unique(y)) > 1 else np.nan,
        pr_auc=average_precision_score(y,p) if len(np.unique(y)) > 1 else np.nan,
        tn=int(tn), fp=int(fp), fn=int(fn), tp=int(tp), alert_rate=float((pred==1).mean()),
        false_alerts_per_10k=float(fp/n*10000), missed_frauds_per_10k=float(fn/n*10000),
        expected_cost=float(fp*fp_cost + fn*fn_cost), expected_cost_per_10k=float((fp*fp_cost + fn*fn_cost)/n*10000))

def choose_threshold(y, p, fn_cost=PRIMARY_FN_COST):
    rows = [metrics(y, p, t, fn_cost=fn_cost) for t in np.round(np.linspace(.01, .90, 90), 2)]
    tab = pd.DataFrame(rows)
    best = tab.sort_values(["expected_cost", "missed_frauds_per_10k", "false_alerts_per_10k", "mcc"], ascending=[True, True, True, False]).iloc[0]
    return float(best.threshold), tab

def topk(y, p):
    y = np.asarray(y).astype(int); p = np.asarray(p); order = np.argsort(-p); total = max(int(y.sum()), 1)
    rows = []
    for pct in TOP_K_PCTS:
        k = max(1, int(math.ceil(len(y)*pct))); idx = order[:k]; found = int(y[idx].sum())
        rows.append(dict(top_k_pct=pct, reviewed_transactions=k, frauds_found=found,
                         recall_at_k=found/total, precision_at_k=found/k))
    return pd.DataFrame(rows)

def latency(model, X):
    n = min(LATENCY_BATCH_SIZE, X.shape[0]); Xb = X[:n]
    for _ in range(LATENCY_WARMUP_RUNS): _ = proba(model, Xb)
    vals = []
    for _ in range(LATENCY_REPEATS):
        t0 = time.perf_counter(); _ = proba(model, Xb); vals.append((time.perf_counter()-t0)/n*1_000_000)
    a = np.array(vals)
    return dict(batch_size=n, latency_mean_us=float(a.mean()), latency_median_us=float(np.median(a)),
                latency_p95_us=float(np.percentile(a,95)), latency_std_us=float(a.std()))

def boot_ci(y, p, thr):
    rng = np.random.default_rng(RANDOM_STATE); y = np.asarray(y).astype(int); p = np.asarray(p); n = len(y)
    vals = {k: [] for k in ["recall", "precision", "mcc", "pr_auc", "roc_auc", "expected_cost_per_10k"]}
    for _ in range(BOOTSTRAP_ROUNDS):
        idx = rng.integers(0, n, n); yy, pp = y[idx], p[idx]
        if len(np.unique(yy)) < 2: continue
        m = metrics(yy, pp, thr)
        for k in vals: vals[k].append(m[k])
    base = metrics(y, p, thr); rows = []
    for k, v in vals.items():
        arr = np.asarray(v)
        rows.append(dict(metric=k, estimate=base[k], ci_low=float(np.percentile(arr,2.5)), ci_high=float(np.percentile(arr,97.5)), bootstrap_rounds_used=len(arr)))
    return pd.DataFrame(rows)

def ece(y, p, bins=10):
    y = np.asarray(y).astype(int); p = np.asarray(p); edges = np.linspace(0, 1, bins+1); out = 0
    for i in range(bins):
        mask = (p >= edges[i]) & ((p < edges[i+1]) if i < bins-1 else (p <= edges[i+1]))
        if mask.any(): out += mask.mean() * abs(y[mask].mean() - p[mask].mean())
    return float(out)

## 8. Train Strong Baselines

In [ ]:
def spw(y):
    y = np.asarray(y).astype(int); return max((y==0).sum(), 1) / max((y==1).sum(), 1)

def xgb_params(scale_pos_weight=None):
    p = dict(n_estimators=300, max_depth=6, learning_rate=.05, subsample=.85, colsample_bytree=.85,
             eval_metric="logloss", random_state=RANDOM_STATE, n_jobs=-1, tree_method="hist")
    if scale_pos_weight is not None: p["scale_pos_weight"] = scale_pos_weight
    return p

def fit_time(model, X, y, label):
    t0 = time.perf_counter(); model.fit(X, y); sec = time.perf_counter() - t0
    print(f"{label:30s} {sec:8.2f}s"); return model, sec

all_models, training_rows = {}, []
for name, ds in prepared.items():
    print("\n" + "="*80 + "\nTraining " + name + "\n" + "="*80)
    specs = [
        ("LR_balanced", LogisticRegression(max_iter=1000, class_weight="balanced", random_state=RANDOM_STATE, n_jobs=-1), "original"),
        ("RF", RandomForestClassifier(n_estimators=200, class_weight="balanced_subsample", random_state=RANDOM_STATE, n_jobs=-1), "original"),
        ("BRF", BalancedRandomForestClassifier(n_estimators=200, random_state=RANDOM_STATE, n_jobs=-1), "original"),
        ("EasyEnsemble", EasyEnsembleClassifier(n_estimators=10, random_state=RANDOM_STATE, n_jobs=-1), "original"),
        ("XGB", xgb.XGBClassifier(**xgb_params()), "original"),
        ("XGB_spw", xgb.XGBClassifier(**xgb_params(spw(ds["y_fit"]))), "original"),
        ("XGB_SMOTE", xgb.XGBClassifier(**xgb_params()), "SMOTE"),
        ("XGB_ADASYN", xgb.XGBClassifier(**xgb_params()), "ADASYN"),
        ("LightGBM", lgb.LGBMClassifier(n_estimators=500, learning_rate=.04, num_leaves=63, class_weight="balanced", random_state=RANDOM_STATE, n_jobs=-1, verbose=-1), "original"),
        ("CatBoost", CatBoostClassifier(iterations=500, learning_rate=.04, depth=6, loss_function="Logloss", eval_metric="AUC", auto_class_weights="Balanced", random_seed=RANDOM_STATE, verbose=False, allow_writing_files=False), "original"),
    ]
    if RUN_SMOTE_TOMEK and "SMOTETomek" in resampled[name]:
        specs.append(("XGB_SMOTETomek", xgb.XGBClassifier(**xgb_params()), "SMOTETomek"))
    all_models[name] = {}
    for mname, model, source in specs:
        if source not in resampled[name]: continue
        Xtr, ytr = resampled[name][source]
        try:
            model, sec = fit_time(model, Xtr, ytr, MODEL_DISPLAY[mname])
            all_models[name][mname] = dict(model=model, train_time_s=sec, train_source=source)
            status = "ok"
        except Exception as e:
            sec, status = np.nan, f"failed: {e}"
            print("FAILED", mname, e)
        training_rows.append(dict(dataset=name, model=mname, model_display=MODEL_DISPLAY[mname], train_source=source, train_time_s=sec, status=status))
    gc.collect()

pd.DataFrame(training_rows).to_csv(TABLE_DIR / "05_training_times.csv", index=False)
pd.DataFrame(training_rows)

## 9. Optional T4 GPU MLP + Focal Loss Baselines

In [ ]:
def dense32(X):
    return X.toarray().astype(np.float32) if sparse.issparse(X) else np.asarray(X, dtype=np.float32)

if RUN_DEEP_MLP and TORCH_OK:
    DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print("MLP device:", DEVICE)

    class FraudMLP(nn.Module):
        def __init__(self, n):
            super().__init__()
            self.net = nn.Sequential(nn.Linear(n,128), nn.BatchNorm1d(128), nn.ReLU(), nn.Dropout(.3),
                                     nn.Linear(128,64), nn.BatchNorm1d(64), nn.ReLU(), nn.Dropout(.3),
                                     nn.Linear(64,32), nn.ReLU(), nn.Linear(32,1))
        def forward(self, x): return self.net(x).squeeze(1)

    class FocalLoss(nn.Module):
        def __init__(self, alpha=.25, gamma=2): super().__init__(); self.alpha=alpha; self.gamma=gamma
        def forward(self, logits, y):
            bce = nn.functional.binary_cross_entropy_with_logits(logits, y.float(), reduction="none")
            pr = torch.sigmoid(logits); pt = torch.where(y == 1, pr, 1-pr)
            at = torch.where(y == 1, torch.tensor(self.alpha, device=y.device), torch.tensor(1-self.alpha, device=y.device))
            return (at * (1-pt).pow(self.gamma) * bce).mean()

    class TorchWrap:
        def __init__(self, model): self.model = model
        def predict_proba(self, X):
            Xd = dense32(X); out = []
            self.model.eval()
            with torch.no_grad():
                for i in range(0, len(Xd), 8192):
                    xb = torch.from_numpy(Xd[i:i+8192]).to(DEVICE)
                    out.append(torch.sigmoid(self.model(xb)).cpu().numpy())
            p = np.concatenate(out); return np.column_stack([1-p, p])

    def train_mlp(Xtr, ytr, Xv, yv, label, epochs=40, batch=2048, patience=6):
        Xtr, Xv = dense32(Xtr), dense32(Xv)
        ytr, yv = np.asarray(ytr, dtype=np.float32), np.asarray(yv, dtype=np.float32)
        loader = DataLoader(TensorDataset(torch.from_numpy(Xtr), torch.from_numpy(ytr)), batch_size=batch, shuffle=True, pin_memory=(DEVICE.type=="cuda"))
        model = FraudMLP(Xtr.shape[1]).to(DEVICE); loss_fn = FocalLoss(); opt = optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)
        Xv_t, yv_t = torch.from_numpy(Xv).to(DEVICE), torch.from_numpy(yv).to(DEVICE)
        best, best_loss, stale, hist = None, 1e99, 0, []; t0 = time.perf_counter()
        for ep in range(epochs):
            model.train(); losses = []
            for xb, yb in loader:
                xb, yb = xb.to(DEVICE), yb.to(DEVICE); opt.zero_grad(set_to_none=True)
                loss = loss_fn(model(xb), yb); loss.backward(); opt.step(); losses.append(float(loss.item()))
            model.eval()
            with torch.no_grad():
                val_loss = float(loss_fn(model(Xv_t), yv_t).item())
            hist.append(dict(epoch=ep+1, train_loss=float(np.mean(losses)), val_loss=val_loss))
            if val_loss < best_loss:
                best_loss = val_loss; best = {k:v.detach().cpu().clone() for k,v in model.state_dict().items()}; stale = 0
            else:
                stale += 1
                if stale >= patience: break
        if best: model.load_state_dict(best)
        sec = time.perf_counter()-t0; print(f"{label:30s} {sec:8.2f}s, epochs={len(hist)}")
        return TorchWrap(model), sec, pd.DataFrame(hist)

    mlp_hist = []
    for name, ds in prepared.items():
        for mname, source in [("MLP_Focal", "original"), ("MLP_Focal_SMOTE", "SMOTE")]:
            try:
                Xtr, ytr = resampled[name][source]
                model, sec, hist = train_mlp(Xtr, ytr, ds["X_val"], ds["y_val"], MODEL_DISPLAY[mname])
                all_models[name][mname] = dict(model=model, train_time_s=sec, train_source=source)
                training_rows.append(dict(dataset=name, model=mname, model_display=MODEL_DISPLAY[mname], train_source=source, train_time_s=sec, status="ok"))
                hist.insert(0, "model", mname); hist.insert(0, "dataset", name); mlp_hist.append(hist)
            except Exception as e:
                print("MLP failed", name, mname, e)
                training_rows.append(dict(dataset=name, model=mname, model_display=MODEL_DISPLAY[mname], train_source=source, train_time_s=np.nan, status=f"failed: {e}"))
            if torch.cuda.is_available(): torch.cuda.empty_cache()
    pd.DataFrame(training_rows).to_csv(TABLE_DIR / "05_training_times.csv", index=False)
    if mlp_hist: pd.concat(mlp_hist).to_csv(TABLE_DIR / "06_mlp_training_history.csv", index=False)
else:
    print("Skipping MLP. RUN_DEEP_MLP=", RUN_DEEP_MLP, "TORCH_OK=", TORCH_OK)

## 10. Validation Thresholds, Final Test Metrics, Top-k, and Latency

In [ ]:
threshold_rows, result_rows, topk_rows, latency_rows = [], [], [], []
pred_store, best_thr = {}, {}
for dname, models in all_models.items():
    ds = prepared[dname]; pred_store[dname], best_thr[dname] = {}, {}
    print("\nEvaluating", dname)
    for mname, obj in models.items():
        model = obj["model"]
        val_p = proba(model, ds["X_val"])
        thr, sweep = choose_threshold(ds["y_val"], val_p)
        best_thr[dname][mname] = thr
        sweep.insert(0, "model_display", MODEL_DISPLAY[mname]); sweep.insert(0, "model", mname); sweep.insert(0, "dataset", dname)
        sweep["selected_threshold"] = sweep.threshold.eq(thr); threshold_rows.append(sweep)
        test_p = proba(model, ds["X_test"]); pred_store[dname][mname] = test_p
        for policy, t in [("default_0.50", .5), ("validation_cost_tuned", thr)]:
            row = metrics(ds["y_test"], test_p, t); row.update(dataset=dname, model=mname, model_display=MODEL_DISPLAY[mname],
                threshold_policy=policy, train_source=obj["train_source"], train_time_s=obj["train_time_s"]); result_rows.append(row)
        tk = topk(ds["y_test"], test_p); tk.insert(0, "model_display", MODEL_DISPLAY[mname]); tk.insert(0, "model", mname); tk.insert(0, "dataset", dname); topk_rows.append(tk)
        try:
            lat = latency(model, ds["X_test"]); lat.update(dataset=dname, model=mname, model_display=MODEL_DISPLAY[mname]); latency_rows.append(lat)
        except Exception as e:
            latency_rows.append(dict(dataset=dname, model=mname, model_display=MODEL_DISPLAY[mname], error=str(e)))
        print(f"{MODEL_DISPLAY[mname]:28s} threshold={thr:.2f}")

threshold_table = pd.concat(threshold_rows, ignore_index=True)
results = pd.DataFrame(result_rows)
topk_table = pd.concat(topk_rows, ignore_index=True)
latency_table = pd.DataFrame(latency_rows)
threshold_table.to_csv(TABLE_DIR / "07_validation_threshold_cost_sweep.csv", index=False)
pd.DataFrame([dict(dataset=d, model=m, model_display=MODEL_DISPLAY[m], selected_threshold=t) for d, mm in best_thr.items() for m,t in mm.items()]).to_csv(TABLE_DIR / "08_selected_thresholds.csv", index=False)
results.to_csv(TABLE_DIR / "09_test_results_default_and_cost_tuned.csv", index=False)
topk_table.to_csv(TABLE_DIR / "10_topk_alert_budget_metrics.csv", index=False)
latency_table.to_csv(TABLE_DIR / "11_repeated_latency_protocol.csv", index=False)
results[results.threshold_policy=="validation_cost_tuned"].sort_values(["dataset","mcc"], ascending=[True,False]).head(30)

## 11. Cost Sensitivity, Bootstrap CI, Calibration, and McNemar-Holm

In [ ]:
cost_rows = []
for dname, ds in prepared.items():
    for mname, p in pred_store[dname].items():
        val_p = proba(all_models[dname][mname]["model"], ds["X_val"])
        for c in FN_COSTS:
            thr, _ = choose_threshold(ds["y_val"], val_p, fn_cost=c)
            row = metrics(ds["y_test"], p, thr, fn_cost=c); row.update(dataset=dname, model=mname, model_display=MODEL_DISPLAY[mname], fn_cost=c, fp_cost=FP_COST)
            cost_rows.append(row)
cost_table = pd.DataFrame(cost_rows); cost_table.to_csv(TABLE_DIR / "12_cost_sensitivity_results.csv", index=False)

ci_rows = []
for dname, ds in prepared.items():
    for mname in [m for m in ["XGB","XGB_spw","XGB_SMOTE","XGB_ADASYN","LightGBM","CatBoost","BRF","EasyEnsemble","MLP_Focal_SMOTE"] if m in pred_store[dname]]:
        ci = boot_ci(ds["y_test"], pred_store[dname][mname], best_thr[dname][mname])
        ci.insert(0, "threshold", best_thr[dname][mname]); ci.insert(0, "model_display", MODEL_DISPLAY[mname]); ci.insert(0, "model", mname); ci.insert(0, "dataset", dname)
        ci_rows.append(ci)
ci_table = pd.concat(ci_rows, ignore_index=True); ci_table.to_csv(TABLE_DIR / "13_bootstrap_95ci.csv", index=False)

cal_rows = []
for dname, ds in prepared.items():
    for mname, p in pred_store[dname].items():
        cal_rows.append(dict(dataset=dname, model=mname, model_display=MODEL_DISPLAY[mname],
                             brier_score=brier_score_loss(ds["y_test"], p), ece_10bin=ece(ds["y_test"], p)))
calibration_table = pd.DataFrame(cal_rows); calibration_table.to_csv(TABLE_DIR / "14_calibration_metrics.csv", index=False)

mc_rows = []
for dname, ds in prepared.items():
    if MAIN_MODEL not in pred_store[dname]: continue
    y = np.asarray(ds["y_test"]).astype(int); main = (pred_store[dname][MAIN_MODEL] >= best_thr[dname][MAIN_MODEL]).astype(int)
    for mname, p in pred_store[dname].items():
        if mname == MAIN_MODEL: continue
        pred = (p >= best_thr[dname][mname]).astype(int)
        a = int(((main==y)&(pred==y)).sum()); b = int(((main==y)&(pred!=y)).sum())
        c = int(((main!=y)&(pred==y)).sum()); d = int(((main!=y)&(pred!=y)).sum())
        pv = float(mcnemar([[a,b],[c,d]], exact=True).pvalue)
        mc_rows.append(dict(dataset=dname, main_model=MAIN_MODEL, comparison_model=mname,
                            comparison_model_display=MODEL_DISPLAY[mname], both_correct=a,
                            main_correct_other_wrong=b, main_wrong_other_correct=c, both_wrong=d, p_value=pv))
mcnemar_table = pd.DataFrame(mc_rows)
if len(mcnemar_table):
    reject, padj, _, _ = multipletests(mcnemar_table.p_value, alpha=.05, method="holm")
    mcnemar_table["p_value_holm"] = padj; mcnemar_table["significant_holm_0_05"] = reject
mcnemar_table.to_csv(TABLE_DIR / "15_mcnemar_holm_results.csv", index=False)
mcnemar_table.head(20)

## 12. Temporal Drift and Drift Severity

In [ ]:
def psi_num(a, b, bins=10):
    a = pd.Series(a).replace([np.inf,-np.inf], np.nan).dropna(); b = pd.Series(b).replace([np.inf,-np.inf], np.nan).dropna()
    if len(a)<20 or len(b)<20 or a.nunique()<2: return np.nan
    edges = np.unique(np.quantile(a, np.linspace(0,1,bins+1)))
    if len(edges)<3: return np.nan
    ac,_ = np.histogram(a, edges); bc,_ = np.histogram(b, edges)
    ap = np.maximum(ac/max(ac.sum(),1), 1e-6); bp = np.maximum(bc/max(bc.sum(),1), 1e-6)
    return float(np.sum((bp-ap)*np.log(bp/ap)))

drift_rows, drift_feature_rows = [], []
for dname, ds in prepared.items():
    raw, label, tcol = ds["raw_for_time"].copy(), ds["label"], ds["time_col"]
    if tcol is None or tcol not in raw.columns:
        print("Skipping drift:", dname); continue
    raw[label] = raw[label].astype(int); raw = add_datetime_features(raw).sort_values(tcol).reset_index(drop=True)
    n = len(raw); tr, va = raw.iloc[:int(.64*n)].copy(), raw.iloc[int(.64*n):int(.80*n)].copy(); te = raw.iloc[int(.80*n):].copy()
    if tr[label].nunique()<2 or va[label].nunique()<2 or te[label].nunique()<2:
        print("Skipping drift due one-class split:", dname); continue
    Xtr, Xv, Xte = tr.drop(columns=[label]), va.drop(columns=[label]), te.drop(columns=[label])
    ytr, yv, yte = tr[label].astype(int), va[label].astype(int), te[label].astype(int)
    Xtr, Xv, Xte, _ = drop_problem_columns(Xtr, Xv, Xte)
    prep = make_preprocessor(Xtr); Xtrp, Xvp, Xtep = as_f32(prep.fit_transform(Xtr)), as_f32(prep.transform(Xv)), as_f32(prep.transform(Xte))
    Xs, ys = SMOTE(random_state=RANDOM_STATE, k_neighbors=5).fit_resample(Xtrp, ytr)
    model, sec = fit_time(xgb.XGBClassifier(**xgb_params()), as_f32(Xs), pd.Series(ys).astype(int), f"Temporal {dname}")
    val_p = proba(model, Xvp); thr, _ = choose_threshold(yv, val_p); test_p = proba(model, Xtep)
    row = metrics(yte, test_p, thr); row.update(dataset=dname, time_col=tcol, train_rows=len(ytr), validation_rows=len(yv),
        test_rows=len(yte), train_fraud_rate_pct=float(ytr.mean()*100), validation_fraud_rate_pct=float(yv.mean()*100),
        test_fraud_rate_pct=float(yte.mean()*100), train_time_s=sec)
    drift_rows.append(row)
    for col in [c for c in tr.select_dtypes(include=["number","bool"]).columns if c != label][:40]:
        try:
            drift_feature_rows.append(dict(dataset=dname, feature=col, psi=psi_num(tr[col], te[col]), ks_statistic=float(ks_2samp(tr[col].dropna(), te[col].dropna()).statistic)))
        except Exception:
            pass
drift_table = pd.DataFrame(drift_rows); drift_feature_table = pd.DataFrame(drift_feature_rows)
drift_table.to_csv(TABLE_DIR / "16_temporal_drift_results.csv", index=False)
drift_feature_table.to_csv(TABLE_DIR / "17_feature_drift_psi_ks.csv", index=False)
drift_table

## 13. Publication-Ready Figures

In [ ]:
plot_results = results[results.threshold_policy=="validation_cost_tuned"].copy()
key = [m for m in ["XGB","XGB_spw","XGB_SMOTE","XGB_ADASYN","LightGBM","CatBoost","BRF","EasyEnsemble","MLP_Focal_SMOTE"] if m in plot_results.model.unique()]
perf = plot_results[plot_results.model.isin(key)].melt(id_vars=["dataset","model","model_display"], value_vars=["recall","mcc","pr_auc"], var_name="metric", value_name="score")
plt.figure(figsize=(12,6.2)); sns.barplot(data=perf, x="model_display", y="score", hue="metric")
plt.xlabel(""); plt.ylabel("Score"); plt.ylim(0,1.05); plt.title("Validation-tuned test performance across stronger baselines")
plt.xticks(rotation=35, ha="right"); plt.legend(title="Metric", ncol=3, loc="upper center", bbox_to_anchor=(.5,1.12))
plt.tight_layout(); savefig("04_model_performance_recall_mcc_prauc"); plt.show()

fig, axes = plt.subplots(1, len(prepared), figsize=(15,4.6), sharey=True)
if len(prepared)==1: axes=[axes]
for ax, (dname, ds) in zip(axes, prepared.items()):
    for m in key:
        if m not in pred_store[dname]: continue
        pr, rc, _ = precision_recall_curve(ds["y_test"], pred_store[dname][m])
        ap = average_precision_score(ds["y_test"], pred_store[dname][m])
        ax.plot(rc, pr, lw=2.4 if m==MAIN_MODEL else 1.2, label=f"{MODEL_DISPLAY[m]} ({ap:.3f})")
    ax.set_title(dname); ax.set_xlabel("Recall")
axes[0].set_ylabel("Precision"); axes[-1].legend(fontsize=6, loc="lower left", bbox_to_anchor=(1.02,0))
plt.suptitle("Precision-recall curves on naturally imbalanced test sets", y=1.04)
plt.tight_layout(); savefig("05_precision_recall_curves"); plt.show()

cost_plot = cost_table[cost_table.model.isin(["XGB","XGB_spw","XGB_SMOTE","LightGBM","CatBoost","BRF","EasyEnsemble"])]
g = sns.catplot(data=cost_plot, x="fn_cost", y="expected_cost_per_10k", hue="model_display", col="dataset", kind="point", height=4.2, aspect=1.05, sharey=False)
g.set_axis_labels("False-negative cost ratio", "Expected cost per 10,000"); g.set_titles("{col_name}")
g.fig.suptitle("Cost-sensitive operating performance", y=1.05); g.fig.tight_layout()
g.fig.savefig(FIG_DIR/"06_cost_sensitive_performance.png", dpi=300, bbox_inches="tight"); g.fig.savefig(FIG_DIR/"06_cost_sensitive_performance.pdf", bbox_inches="tight"); plt.show()

tk = topk_table[topk_table.model.isin(["XGB","XGB_spw","XGB_SMOTE","LightGBM","CatBoost","BRF","EasyEnsemble"])].copy()
tk["top_k_percent"] = tk.top_k_pct * 100
g = sns.relplot(data=tk, x="top_k_percent", y="recall_at_k", hue="model_display", col="dataset", kind="line", marker="o", height=4.2, aspect=1.05)
g.set_axis_labels("Transactions reviewed (%)", "Fraud recall captured"); g.set_titles("{col_name}")
g.fig.suptitle("Alert-budget performance for investigator workload", y=1.05); g.fig.tight_layout()
g.fig.savefig(FIG_DIR/"07_topk_alert_budget_recall.png", dpi=300, bbox_inches="tight"); g.fig.savefig(FIG_DIR/"07_topk_alert_budget_recall.pdf", bbox_inches="tight"); plt.show()

lat = latency_table[latency_table.model.isin(key) & latency_table.latency_p95_us.notna()]
plt.figure(figsize=(9,5.5)); sns.barplot(data=lat, y="model_display", x="latency_p95_us", hue="dataset", orient="h")
plt.xlabel("p95 inference latency per transaction (microseconds)"); plt.ylabel(""); plt.title("Repeated-run inference latency")
plt.tight_layout(); savefig("08_latency_p95_microseconds"); plt.show()

if len(drift_table) and "recall" in drift_table.columns:
    dl = drift_table.melt(id_vars=["dataset"], value_vars=["recall","mcc","pr_auc"], var_name="metric", value_name="score")
    plt.figure(figsize=(8.5,4.8)); sns.barplot(data=dl, x="dataset", y="score", hue="metric")
    plt.ylim(0,1.05); plt.xlabel(""); plt.ylabel("Score"); plt.title("Temporal validation on future transactions")
    plt.xticks(rotation=15, ha="right"); plt.tight_layout(); savefig("09_temporal_drift_performance"); plt.show()

fig, axes = plt.subplots(1, len(prepared), figsize=(15,4.6), sharey=True)
if len(prepared)==1: axes=[axes]
for ax, (dname, ds) in zip(axes, prepared.items()):
    for m in [x for x in ["XGB","XGB_spw","XGB_SMOTE","LightGBM","CatBoost"] if x in pred_store[dname]]:
        pt, pp = calibration_curve(ds["y_test"], pred_store[dname][m], n_bins=10, strategy="quantile")
        ax.plot(pp, pt, marker="o", lw=1.3, label=MODEL_DISPLAY[m])
    ax.plot([0,1],[0,1],"--", color="black", lw=1); ax.set_title(dname); ax.set_xlabel("Mean predicted probability")
axes[0].set_ylabel("Observed fraud rate"); axes[-1].legend(fontsize=6, loc="lower left", bbox_to_anchor=(1.02,0))
plt.suptitle("Calibration curves", y=1.04); plt.tight_layout(); savefig("10_calibration_curves"); plt.show()

if len(mcnemar_table):
    heat = mcnemar_table.pivot(index="comparison_model_display", columns="dataset", values="p_value_holm")
    plt.figure(figsize=(8, max(4, .35*len(heat)))); sns.heatmap(heat, annot=True, fmt=".3g", cmap="viridis_r", cbar_kws={"label":"Holm-adjusted p-value"})
    plt.title("McNemar tests vs XGBoost + SMOTE"); plt.xlabel(""); plt.ylabel("Comparison model")
    plt.tight_layout(); savefig("11_mcnemar_holm_heatmap"); plt.show()

## 14. SHAP Explainability

In [ ]:
shap_rows, local_rows = [], []
for dname, ds in prepared.items():
    if MAIN_MODEL not in all_models[dname]: continue
    model = all_models[dname][MAIN_MODEL]["model"]; Xte = ds["X_test"]; y = np.asarray(ds["y_test"]).astype(int); names = ds["feature_names"]
    idx = np.random.default_rng(RANDOM_STATE).choice(Xte.shape[0], size=min(SHAP_SAMPLE_SIZE, Xte.shape[0]), replace=False)
    Xs = Xte[idx]
    try:
        explainer = shap.TreeExplainer(model); sv = np.asarray(explainer.shap_values(Xs))
        if sv.ndim == 3: sv = sv[:, :, -1]
        mean_abs = np.abs(sv).mean(axis=0); order = np.argsort(mean_abs)[::-1][:15]
        for r, j in enumerate(order, 1):
            shap_rows.append(dict(dataset=dname, rank=r, feature=names[j] if j < len(names) else f"feature_{j}", mean_abs_shap=float(mean_abs[j])))
        sdf = pd.DataFrame([x for x in shap_rows if x["dataset"]==dname]).sort_values("mean_abs_shap")
        plt.figure(figsize=(7,5)); sns.barplot(data=sdf, x="mean_abs_shap", y="feature", color="#4C78A8")
        plt.xlabel("Mean absolute SHAP value"); plt.ylabel(""); plt.title(f"SHAP global feature importance - {dname}")
        plt.tight_layout(); savefig(f"12_shap_global_{dname}"); plt.show()

        p = pred_store[dname][MAIN_MODEL]; thr = best_thr[dname][MAIN_MODEL]; pred = (p >= thr).astype(int)
        for case, cand in {"true_positive": np.where((y==1)&(pred==1))[0], "false_positive": np.where((y==0)&(pred==1))[0]}.items():
            if len(cand) == 0: continue
            row = int(cand[0]); X1 = Xte[row:row+1]; sv1 = np.asarray(explainer.shap_values(X1))
            if sv1.ndim == 3: sv1 = sv1[:, :, -1]
            sv1 = sv1.reshape(-1); vals = X1.toarray().reshape(-1) if sparse.issparse(X1) else np.asarray(X1).reshape(-1)
            top = np.argsort(np.abs(sv1))[::-1][:10]
            for r, j in enumerate(top, 1):
                local_rows.append(dict(dataset=dname, case_type=case, test_row_index=row, rank=r,
                    feature=names[j] if j < len(names) else f"feature_{j}", feature_value=float(vals[j]) if np.isfinite(vals[j]) else np.nan,
                    shap_value=float(sv1[j]), predicted_probability=float(p[row]), threshold=float(thr)))
            ldf = pd.DataFrame([x for x in local_rows if x["dataset"]==dname and x["case_type"]==case]).sort_values("shap_value")
            plt.figure(figsize=(7,4.6)); colors = ["#D55E00" if v>0 else "#0072B2" for v in ldf.shap_value]
            plt.barh(ldf.feature, ldf.shap_value, color=colors); plt.axvline(0, color="black", lw=.8)
            plt.xlabel("SHAP contribution to fraud score"); plt.ylabel(""); plt.title(f"Local SHAP: {case.replace('_',' ')} - {dname}")
            plt.tight_layout(); savefig(f"13_shap_local_{case}_{dname}"); plt.show()
    except Exception as e:
        print("SHAP failed", dname, e)

shap_global_table = pd.DataFrame(shap_rows); local_explanation_table = pd.DataFrame(local_rows)
shap_global_table.to_csv(TABLE_DIR / "18_shap_global_feature_importance.csv", index=False)
local_explanation_table.to_csv(TABLE_DIR / "19_shap_local_explanations.csv", index=False)
shap_global_table.head(20)

## 15. Save Models, Manuscript Summary, and Output Manifest

In [ ]:
for dname, models in all_models.items():
    mdir = MODEL_DIR / dname; mdir.mkdir(exist_ok=True)
    joblib.dump(prepared[dname]["preprocessor"], mdir / "preprocessor.joblib")
    for mname, obj in models.items():
        if mname.startswith("MLP"): continue
        try: joblib.dump(obj["model"], mdir / f"{mname}.joblib")
        except Exception as e: print("Could not save", dname, mname, e)

summary = results[results.threshold_policy=="validation_cost_tuned"].sort_values(["dataset","mcc"], ascending=[True,False])
cols = ["dataset","model","model_display","threshold","recall","precision","f1","mcc","roc_auc","pr_auc",
        "false_alerts_per_10k","missed_frauds_per_10k","expected_cost_per_10k","train_time_s"]
summary[cols].to_csv(TABLE_DIR / "20_manuscript_model_summary.csv", index=False)

manifest = []
for folder in [TABLE_DIR, FIG_DIR, MODEL_DIR, LOG_DIR]:
    for p in sorted(folder.rglob("*")):
        if p.is_file(): manifest.append(dict(folder=folder.name, file=p.name, path=str(p), bytes=p.stat().st_size))
manifest_df = pd.DataFrame(manifest)
manifest_df.to_csv(OUT_DIR / "00_output_manifest.csv", index=False)
print("Output folder:", OUT_DIR)
print("Use tables 01-20 and figures 01-13 in the paper.")
manifest_df.tail(30)